# At a glance

In [130]:
#| output: false
#| eval: true
#| echo: false

# Load Excel sheet
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

filename = "../private/CARD Group Timeline.xlsx"

df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

names = df['Display Name'].values

# Get the current group members and alumni
current_group_member_indices = df[df.apply( \
    lambda row: row.astype(str).str.contains('current').any(),
      axis=1)].index.tolist()
alumni_indices = df[~df.index.isin(current_group_member_indices)].index.tolist()

current_group_members = df.loc[current_group_member_indices, 'Display Name']
alumni = df.loc[alumni_indices, 'Display Name']

current_index = np.zeros([len(names), 1])
current_index[current_group_member_indices] = 1
alumni_index = np.zeros([len(names), 1])
alumni_index[alumni_indices] = 1
df['current'] = current_index
df['alumni'] = alumni_index
del current_index, alumni_index

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



In [131]:
#| echo: false
#| output: true
#| eval: true

import plotly.graph_objects as go
import numpy as np
from IPython.display import HTML

def make_sparkline(data, color="white"):
    """Return Plotly sparkline HTML."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=data, mode="lines", line=dict(color=color, width=2)
    ))
    fig.update_layout(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=0, r=0, t=0, b=0),
        height=80,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)"
    )
    return fig.to_html(include_plotlyjs=False, full_html=False)

def make_value_box(title, value, icon_name, theme, decimals=1):
    """Return HTML for a value box with an icon."""
    return f"""
    <div class="value-box {theme}">
      <div class="top-row">
        <div class="label">{title}</div>
        <iconify-icon icon="{icon_name}"></iconify-icon>
      </div>
      <div class="value">{value:.{decimals}f}</div>
    </div>
    """


import datetime

dt1 = datetime.datetime(year=2022, month=8, day=15)
dt2 = datetime.datetime.now()
eltime = dt2 - dt1
#print("{0:2.1f} years.".format(eltime.days / 365.25))

total = df[df['current']==True]['Display Name'].count()

#print("{0} students currently working towards degrees.".format(total))


grad = df[df['alumni']==True]['Ultimate Degree-Role'].str.contains('PhD').sum() + df[df['alumni']==True]['Ultimate Degree-Role'].str.contains('MS').sum() + df[df['alumni']==True]['Ultimate Degree-Role'].str.contains('MEng').sum()


# Build multiple boxes
boxes_html = f"""
<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(450px,1fr));gap:1rem;">
  {make_value_box("Years of activity", eltime.days / 365.25, "fluent-mdl2:calendar-year", "success", decimals=1)}
  {make_value_box("Students currently working toward degrees", total, "fa-solid:user-graduate", "info", decimals=0)}
  {make_value_box("Graduate degrees awarded", grad, "vaadin:diploma-scroll", "warning", decimals=0)}
  {make_value_box("Undergraduates mentored in scientific research",
                  df['Ultimate Degree-Role'].str.contains('BS').sum(), "hugeicons:brain-02", "danger", decimals=0)}
</div>
"""

HTML(boxes_html)

In [132]:
#| eval: true
#| echo: false
#| output: false
#| include: false

import pandas as pd
import plotly.express as px
import numpy as np
from plotly.subplots import make_subplots

filename = "../private/CARD Group Timeline.xlsx"

df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

names = df['Display Name'].values

# Get the current group members and alumni
current_group_member_indices = df[df.apply( \
    lambda row: row.astype(str).str.contains('current').any(),
      axis=1)].index.tolist()
alumni_indices = df[~df.index.isin(current_group_member_indices)].index.tolist()

current_group_members = df.loc[current_group_member_indices, 'Display Name']
alumni = df.loc[alumni_indices, 'Display Name']

current_index = np.zeros([len(names), 1])
current_index[current_group_member_indices] = 1
alumni_index = np.zeros([len(names), 1])
alumni_index[alumni_indices] = 1
df['current'] = current_index
df['alumni'] = alumni_index
del current_index, alumni_index


/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



# Breakdown of Current Group Composition by Student Degree Program and Role

In [13]:
#| eval: true
#| echo: false
#| output: true

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML
import textwrap
import colorsys

filename = "../private/CARD Group Timeline.xlsx"
df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

# Mark current members by looking for 'current' in any role finish column/value.
current_mask = df.apply(lambda row: row.astype(str).str.contains("current", case=False, na=False).any(), axis=1)

# Focus on current students and map each role into a multi-level hierarchy.
current_students = df[current_mask].copy()
roles = (
    current_students["Ultimate Degree-Role"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
)


def map_role_to_hierarchy(role):
    role_lower = role.lower()

    if role.startswith("PhD"):
        population = "Graduate"
        degree = "PhD"
        if "primary" in role_lower:
            grouping = "Primary Advisee"
        elif "secondary" in role_lower:
            grouping = "Secondary Advisee"
        else:
            grouping = "Other"
        return population, degree, grouping

    if role.startswith("MS"):
        population = "Graduate"
        degree = "MS"
        if "primary" in role_lower:
            grouping = "Primary Advisee"
        elif "secondary" in role_lower:
            grouping = "Secondary Advisee"
        else:
            grouping = "Other"
        return population, degree, grouping

    if role.startswith("MEng"):
        population = "Graduate"
        degree = "MEng"
        if "primary" in role_lower:
            grouping = "Primary Advisee"
        elif "secondary" in role_lower:
            grouping = "Secondary Advisee"
        else:
            grouping = "Other"
        return population, degree, grouping

    if role.startswith("BS"):
        population = "Undergraduate"
        degree = "BS"
        if "research experience for undergraduates" in role_lower:
            grouping = "REU Participant"
        elif "experiential exploration program" in role_lower:
            grouping = "EEP Researcher"
        elif "experiential education program" in role_lower:
            grouping = "EEP Researcher"
        else:
            grouping = "Other"
        return population, degree, grouping

    return "Other", "Other", role


def wrap_label(text, width=12, force_single_line=False):
    if force_single_line:
        return str(text)
    return "<br>".join(textwrap.wrap(str(text), width=width, break_long_words=False, break_on_hyphens=False))


def hls_to_hex(h, l, s):
    r, g, b = colorsys.hls_to_rgb(h % 1.0, l, s)
    return "#{:02X}{:02X}{:02X}".format(int(r * 255), int(g * 255), int(b * 255))


hierarchy = roles.apply(map_role_to_hierarchy).apply(pd.Series)
hierarchy.columns = ["Population", "Degree", "Grouping"]

root_color = "#1F2937"
population_hues = {
    "Graduate": 0.55,
    "Undergraduate": 0.80,
    "Other": 0.00,
}
degree_hues = {
    "PhD": 0.60,
    "MS": 0.08,
    "MEng": 0.35,
    "BS": 0.78,
    "Other": 0.00,
}
group_hue_shift = {
    "Primary Advisee": -0.03,
    "Secondary Advisee": 0.03,
    "EEP Researcher": 0.07,
    "REU Participant": 0.11,
    "Other": 0.00,
}


def population_color(population):
    return hls_to_hex(population_hues.get(population, 0.00), l=0.38, s=0.58)


def degree_color(degree):
    return hls_to_hex(degree_hues.get(degree, 0.00), l=0.52, s=0.66)


def group_color(degree, grouping):
    base_hue = degree_hues.get(degree, 0.00)
    hue = base_hue + group_hue_shift.get(grouping, 0.00)
    return hls_to_hex(hue, l=0.66, s=0.72)


ids = []
parents = []
labels = []
values = []
colors = []
raw_labels = []


def add_node(node_id, parent_id, label, value, color, force_single_line=False):
    ids.append(node_id)
    parents.append(parent_id)
    wrapped = wrap_label(label, width=12, force_single_line=force_single_line)
    labels.append(f"{wrapped} ({int(value)})")
    values.append(int(value))
    colors.append(color)
    raw_labels.append(label)


root = "CARD Group Students"
add_node(root, "", root, len(hierarchy), root_color, force_single_line=True)

for population in sorted(hierarchy["Population"].unique()):
    pop_df = hierarchy[hierarchy["Population"] == population]
    pop_id = f"{root}|{population}"
    add_node(pop_id, root, population, len(pop_df), population_color(population))

    degrees = sorted(pop_df["Degree"].unique())

    # If there is only one degree under a population, skip that extra level.
    if len(degrees) == 1:
        only_degree = degrees[0]
        deg_df = pop_df[pop_df["Degree"] == only_degree]
        groups = sorted(deg_df["Grouping"].unique())

        # If there is also only one grouping, keep population as the leaf.
        if len(groups) == 1:
            continue

        for grouping in groups:
            grp_df = deg_df[deg_df["Grouping"] == grouping]
            grp_id = f"{pop_id}|{grouping}"
            add_node(grp_id, pop_id, grouping, len(grp_df), group_color(only_degree, grouping))
        continue

    for degree in degrees:
        deg_df = pop_df[pop_df["Degree"] == degree]
        deg_id = f"{pop_id}|{degree}"
        add_node(deg_id, pop_id, degree, len(deg_df), degree_color(degree))

        groups = sorted(deg_df["Grouping"].unique())

        # If there is only one grouping under a degree, skip that extra level.
        if len(groups) == 1:
            continue

        for grouping in groups:
            grp_df = deg_df[deg_df["Grouping"] == grouping]
            grp_id = f"{deg_id}|{grouping}"
            add_node(grp_id, deg_id, grouping, len(grp_df), group_color(degree, grouping))

fig = go.Figure(
    go.Treemap(
        ids=ids,
        parents=parents,
        labels=labels,
        values=values,
        branchvalues="total",
        marker=dict(colors=colors, line=dict(width=1.25, color="white")),
        textinfo="label",
        textfont=dict(size=13, color="white"),
        hovertemplate="%{customdata}<br>Count: %{value}<extra></extra>",
        customdata=raw_labels
    )
)

fig.update_layout(
    title="",
    margin=dict(t=60, l=10, r=10, b=10),
    uniformtext=dict(minsize=9, mode="show")
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



# Group Roles

In [ ]:
#| eval: true
#| echo: false

plots = [
        px.pie(df, names='Ultimate Degree-Role'),
        px.pie(df[df['current']==True], names='Ultimate Degree-Role'),
        px.pie(df[df['alumni']==True], names='Ultimate Degree-Role')
        ]

fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]], subplot_titles=('Since 2022', 'Current', 'Alumni'), horizontal_spacing=0.01)

for i, figure in enumerate(plots):
    for trace in figure.data:
        fig.add_trace(trace, row=1, col=i+1)

# Update the layout to position the legend below the plot
fig.update_layout(
    legend=dict(
        orientation="h",  # Optional: set orientation to horizontal
        yanchor="bottom",    # Anchor the top of the legend to the specified
        y=-0.75,           # Adjust this value to control the vertical position below the plot
        xanchor="center",   # Anchor the left of the legend to the specified x-coordinate
        x=0.5,          # Position the legend at the left edge
        itemsizing='constant', # Ensures consistent legend symbol size
        itemwidth=30, # Sets the width of the legend item (including symbol)
        entrywidthmode="fraction", # Control entry width
        entrywidth=0.5, # Example: each entry takes 50% of legend width, creating two columns
        font=dict(size=8) # Adjust font size if needed for better fit
    )
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

In [ ]:
#| eval: true
#| echo: false
#| output: true

import datetime
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML
import re

# Load data
filename = "../private/CARD Group Timeline.xlsx"
df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

# Load roles and colors from the Roles sheet
roles_df = pd.read_excel(filename, sheet_name="Roles")
role_order = roles_df['Roles'].tolist()

# Map color names to hex codes
color_name_to_hex = {
    'Red': '#d62728',
    'Blue': '#1f77b4',
    'Green': '#2ca02c',
    'Violet': '#9467bd',
    'Dark Blue': '#08519c',
    'Red Stripe': '#ff7f7f',
    'Blue Stripe': '#7fbfff',
    'Green Stripe': '#7fff7f',
    'Red Dot': '#8b3a3a',
    'Blue Dot': '#0f3a7f',
    'Green Dot': '#0f7f0f'
}

role_colors = {}
for role, color_name in zip(roles_df['Roles'], roles_df['Color']):
    role_colors[role] = color_name_to_hex.get(color_name, '#cccccc')

# Helper function to parse dates
def parse_date(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip().lower()
    if text in ['nan', 'nat', 'none', '']:
        return None
    if text == 'current':
        return None
    try:
        return pd.to_datetime(value)
    except:
        return None

# Extract timeline for each person
def get_timeline_events(row):
    events = []
    for level in ['Ultimate', 'Penultimate', 'Antepenultimate', 'Preantepenultimate', 'Propreantepenultimate', 'Ultrasuprapropreantepenultimate']:
        role_col = f'{level} Degree-Role'
        start_col = f'{level} Role Start'
        finish_col = f'{level} Role Finish'
        
        if role_col not in df.columns or start_col not in df.columns:
            continue
        
        role = row.get(role_col)
        if pd.isna(role) or role is None or str(role).strip() == '':
            continue
        
        start = parse_date(row.get(start_col))
        if start is None:
            continue
        
        finish = parse_date(row.get(finish_col))
        if finish is None:
            finish = pd.Timestamp(datetime.date.today())
        
        events.append({
            'role': role,
            'start': start,
            'finish': finish
        })
    
    return events

# Collect all timeline events
timeline_events = []
for idx, row in df.iterrows():
    events = get_timeline_events(row)
    timeline_events.extend(events)

# Get all unique dates and sort them
all_dates = set()
for event in timeline_events:
    all_dates.add(event['start'].date())
    all_dates.add(event['finish'].date())
all_dates = sorted(list(all_dates))

# Count roles on each date
role_counts = {role: [] for role in role_order}
dates_for_chart = []

for date in all_dates:
    dates_for_chart.append(date)
    for role in role_order:
        count = 0
        for event in timeline_events:
            if event['role'] == role and event['start'].date() <= date <= event['finish'].date():
                count += 1
        role_counts[role].append(count)

# Create the stacked line chart
fig = go.Figure()

# Helper function to remove parentheticals from text
def remove_parentheticals(text):
    return re.sub(r'\s*\([^)]*\)', '', text).strip()

# Add traces for each role (in reverse order so the stack builds from bottom to top)
for role in reversed(role_order):
    if all(count == 0 for count in role_counts[role]):
        # Skip roles with no members
        continue
    
    # Remove parentheticals for legend display only
    legend_name = remove_parentheticals(role)
    
    fig.add_trace(go.Scatter(
        x=dates_for_chart,
        y=role_counts[role],
        mode='lines',
        name=legend_name,
        stackgroup='one',
        line=dict(width=2, color=role_colors.get(role, '#cccccc')),
        fillcolor=role_colors.get(role, '#cccccc'),
        fill='tonexty',
        hovertemplate='<b>%{x|%B %Y}</b><br>' + role + ': %{y}<extra></extra>'
    ))

fig.update_layout(
    title='CARD Lab Size Over Time',
    xaxis_title='Date',
    yaxis_title='Number of Members',
    hovermode='closest',
    height=700,
    margin=dict(b=300),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.30,
        xanchor='center',
        x=0.5,
        entrywidthmode='fraction',
        entrywidth=0.5,
        font=dict(size=10)
    )
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



# Nationality

In [136]:
#| eval: true
#| echo: false

filename = "../private/CARD Group Timeline.xlsx"

df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

# Get the current group members and alumni
current_group_member_indices = df[df.apply(
    lambda row: row.astype(str).str.contains('current').any(), axis=1)].index.tolist()
alumni_indices = df[~df.index.isin(current_group_member_indices)].index.tolist()

current_group_members = df.loc[current_group_member_indices, 'Display Name']
alumni = df.loc[alumni_indices, 'Display Name']

current_index = np.zeros([len(names), 1])
current_index[current_group_member_indices] = 1
alumni_index = np.zeros([len(names), 1])
alumni_index[alumni_indices] = 1
df['current'] = current_index
df['alumni'] = alumni_index
del current_index, alumni_index


plots = [
        px.pie(df, names='Nationality'),
        px.pie(df[df['current']==True], names='Nationality'),
        px.pie(df[df['alumni']==True], names='Nationality')
        ]

fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]], subplot_titles=('Since 2022', 'Current', 'Alumni'))

for i, figure in enumerate(plots):
    for trace in figure.data:
        fig.add_trace(trace, row=1, col=i+1)

# Update the layout to position the legend below the plot
fig.update_layout(
    legend=dict(
        orientation="h",  # Optional: set orientation to horizontal
        yanchor="top",    # Anchor the top of the legend to the specified y-coordinate
        y=-0.3,           # Adjust this value to control the vertical position below the plot
        xanchor="left",   # Anchor the left of the legend to the specified x-coordinate
        x=0     ,          # Position the legend at the left edge
        tracegroupgap=10,  # Optional: add some space between legend groups
        # To achieve two columns and word wrap, you'll need to control
        # the overall width and rely on the legend items to wrap naturally
        # if their combined width exceeds the available space.
        # There isn't a direct 'columns=2' property for legends in Plotly.
        # Instead, you control the legend's overall dimensions.
        itemwidth=30,     # Set a fixed width for each legend item
        itemsizing="constant", # Keep item width constant
        font=dict(size=10) # Adjust font size if needed for better fit
    )
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



# Places we have called home

In [134]:
#| eval: true
#| echo: false
#| output: false
#| warning: false
#| error: false

%conda install -y geopy
%pip install -q openlocationcode

import os
import re
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from openlocationcode import openlocationcode as olc
import itertools

filename = "../private/CARD Group Timeline.xlsx"
cache_file = "../private/hometown_coordinates.txt"

df = pd.read_excel(filename, sheet_name="People")

# === Step 1: Define our city data ===
cities = []
for i in range(1, 11):
    cities.append(df["Hometown {0}".format(i)].dropna().values)
cities = list(itertools.chain.from_iterable(cities))

# === Step 2: Set up geocoder and cache helpers ===
geolocator = Nominatim(user_agent="city_mapper")

def clean_location_label(text):
    cleaned = str(text).strip()
    patterns = [
        r"\b\d{5}(?:-\d{4})?\b",  # US ZIP
        r"\b\d{4}\b",  # Australia/New Zealand-style 4-digit postal codes
        r"\b\d{6}\b",  # India/Nigeria/China postal codes
        r"\b[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z][ -]?\d[ABCEGHJ-NPRSTV-Z]\d\b",  # Canada
        r"\b(?:GIR\s?0AA|[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2})\b",  # UK
        r"\b(?=[A-Z0-9-]{3,10}\b)(?=.*\d)[A-Z0-9-]+\b",  # Generic alphanumeric postal token
    ]
    for pattern in patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"\s+,", ",", cleaned)
    cleaned = re.sub(r",\s*,", ", ", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned)
    return cleaned.strip(" ,")

def has_postal_code_token(text):
    return bool(
        re.search(
            r"\b\d{5}(?:-\d{4})?\b|\b\d{6}\b|\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b|\b[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z][ -]?\d[ABCEGHJ-NPRSTV-Z]\d\b",
            str(text),
            flags=re.IGNORECASE,
        )
    )

def should_refresh_cached_label(query, label):
    cleaned_query = clean_location_label(query)
    cleaned_label = clean_location_label(label)

    if not cleaned_label:
        return True
    if has_postal_code_token(cleaned_label):
        return True

    # If the original query is detailed but cached label is generic, refresh it.
    if "," in cleaned_query and "," not in cleaned_label:
        first_part = cleaned_query.split(",", 1)[0].strip()
        if first_part and first_part.lower() not in cleaned_label.lower():
            return True

    return False

def geocode_with_retry(address, retries=20, delay=5):
    for attempt in range(retries):
        try:
            return geolocator.geocode(address)
        except GeocoderTimedOut:
            time.sleep(delay)
        except Exception:
            return None
    return None

def parse_plus_code_input(value):
    text = str(value).strip()
    match = re.search(
        r"([23456789CFGHJMPQRVWX]{2,8}\+[23456789CFGHJMPQRVWX]{2,})",
        text,
        flags=re.IGNORECASE,
    )
    if not match:
        return None, None

    code = match.group(1).upper()
    context = text[match.end():].strip(" ,")
    return code, context

def plus_code_coordinates(value):
    code, context = parse_plus_code_input(value)
    if not code:
        return None

    if olc.isValid(code) and olc.isFull(code):
        area = olc.decode(code)
        return {
            "latitude": area.latitudeCenter,
            "longitude": area.longitudeCenter,
            "context_location": None,
        }

    if olc.isValid(code) and olc.isShort(code):
        if not context:
            return None
        context_location = geocode_with_retry(context)
        if not context_location:
            return None

        full_code = olc.recoverNearest(code, context_location.latitude, context_location.longitude)
        area = olc.decode(full_code)
        return {
            "latitude": area.latitudeCenter,
            "longitude": area.longitudeCenter,
            "context_location": context_location,
        }

    return None

def display_name_from_location(location, fallback=""):
    if location is None:
        return clean_location_label(fallback)

    address = (location.raw or {}).get("address", {})
    locality = (
        address.get("city")
        or address.get("town")
        or address.get("village")
        or address.get("municipality")
        or address.get("county")
        or address.get("state")
    )
    state = address.get("state")
    state_code = address.get("state_code")
    country = address.get("country")
    country_code = str(address.get("country_code", "")).lower()
    region = state_code or state

    # Use state for US labels to keep them compact and ZIP-free.
    if country_code == "us":
        if locality and region:
            return clean_location_label(f"{locality}, {region}")
        if locality:
            return clean_location_label(locality)
        if region:
            return clean_location_label(region)

    if locality and region and country:
        return clean_location_label(f"{locality}, {region}, {country}")
    if locality and country:
        return clean_location_label(f"{locality}, {country}")

    fallback_clean = clean_location_label(fallback)
    if (not locality) and ("," in fallback_clean):
        return fallback_clean
    if locality:
        return clean_location_label(str(locality))
    if country:
        return clean_location_label(str(country))
    return clean_location_label(fallback)

def plus_code_label(query, reverse_location=None, context_location=None):
    code, context = parse_plus_code_input(query)
    if context_location is not None:
        # For short plus codes, prefer the explicit locality context supplied with the code.
        return display_name_from_location(context_location, fallback=context)
    if reverse_location is not None:
        return display_name_from_location(reverse_location, fallback="Decoded plus code location")
    return clean_location_label(context or "Decoded plus code location")

def load_coordinate_cache(path):
    coordinate_cache = {}
    label_cache = {}
    if not os.path.exists(path):
        return coordinate_cache, label_cache
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            place, lon, lat = parts[:3]
            label = parts[3] if len(parts) > 3 else ""
            try:
                coordinate_cache[place] = (float(lon), float(lat))
                if label:
                    label_cache[place] = clean_location_label(label)
            except ValueError:
                continue
    return coordinate_cache, label_cache

def save_coordinate_cache(path, coordinate_cache, label_cache):
    with open(path, "w", encoding="utf-8") as f:
        for place, (lon, lat) in coordinate_cache.items():
            label = clean_location_label(label_cache.get(place, ""))
            if label:
                f.write(f"{place}\t{lon}\t{lat}\t{label}\n")
            else:
                f.write(f"{place}\t{lon}\t{lat}\n")

def reverse_with_retry(latitude, longitude, retries=10, delay=3):
    for attempt in range(retries):
        try:
            return geolocator.reverse(
                f"{latitude}, {longitude}",
                exactly_one=True,
                language="en"
            )
        except GeocoderTimedOut:
            time.sleep(delay)
        except Exception:
            return None
    return None

# === Step 3: Geocode only missing locations ===
coordinate_cache, label_cache = load_coordinate_cache(cache_file)
lonlat = []
location_names = []

for city in cities:
    query = str(city).strip(", ")

    if query in coordinate_cache:
        lon, lat = coordinate_cache[query]
        display_name = clean_location_label(label_cache.get(query, ""))

        # Backfill missing, postal-like, or overly generic labels.
        if should_refresh_cached_label(query, display_name):
            query_label = clean_location_label(query)
            if "," in query_label:
                display_name = query_label
            else:
                query_location = geocode_with_retry(query)
                if query_location:
                    display_name = display_name_from_location(query_location, fallback=query)
                else:
                    reverse_location = reverse_with_retry(lat, lon)
                    display_name = display_name_from_location(reverse_location, fallback=query)
            label_cache[query] = display_name

        lonlat.append([lon, lat])
        location_names.append(display_name)
        print(f"Cached: {query} -> {lat}, {lon}")
        continue

    # Decode plus codes directly to coordinates before normal geocoding.
    plus_code_result = plus_code_coordinates(query)
    if plus_code_result:
        lat = plus_code_result["latitude"]
        lon = plus_code_result["longitude"]
        reverse_location = reverse_with_retry(lat, lon)
        display_name = plus_code_label(
            query,
            reverse_location=reverse_location,
            context_location=plus_code_result["context_location"],
        )

        lonlat.append([lon, lat])
        location_names.append(display_name)
        coordinate_cache[query] = (lon, lat)
        label_cache[query] = display_name
        print(f"Plus code decoded: {query} -> {lat}, {lon}")
        continue

    print(f"Geocoding: {query}")
    location = geocode_with_retry(query)
    if location:
        lon, lat = location.longitude, location.latitude
        display_name = display_name_from_location(location, fallback=query)

        lonlat.append([lon, lat])
        location_names.append(display_name)
        coordinate_cache[query] = (lon, lat)
        label_cache[query] = display_name
        print(f"New: {query} -> {lat}, {lon}")
        time.sleep(1)
    else:
        print(f"{query} -> Not found.")

# Persist normalized labels so future runs avoid fallback string parsing.
save_coordinate_cache(cache_file, coordinate_cache, label_cache)

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 26.1.1
  latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda

Or to minimize the number of packages updated during conda update use

     conda install conda=26.3.2



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Cached: El Lago, TX 77586, USA -> 29.5710406, -95.0440812
Cached: Orsu, Imo, Nigeria -> 5.8494268, 6.9828062
Cached: Hanumangarh, Rajasthan 335512, India -> 29.6152878, 74.2903968
Geocoding: Odo-Ona Kekere, Ibadan 200258, Oyo, Nigeria
Odo-Ona Kekere, Ibadan 200258, Oyo, Nigeria -> Not found.
Cached: West Chester Township, OH 45069, USA -> 39.3359319, -84.4155616
Cached: Hilliard, OH 43026, USA -> 40.033814, -83.1596108
Cached: Amberpet, Hyderabad, Telangana 500007, India -> 17.3861776, 78.5114709
Cac

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



In [ ]:
#| eval: true
#| echo: false
#| output: true
#| warning: false
#| error: false

import pandas as pd
import plotly.express as px
from IPython.display import HTML

# Build a DataFrame from cached/geocoded coordinates generated in the previous cell.
locations_df = pd.DataFrame(
    {
        "city": location_names,
        "longitude": [item[0] for item in lonlat],
        "latitude": [item[1] for item in lonlat],
    }
)

fig = px.scatter_geo(
    locations_df,
    lon="longitude",
    lat="latitude",
    hover_name="city",
    hover_data={"longitude": False, "latitude": False},
    title=""
 )

fig.update_traces(marker=dict(size=6, color="red", opacity=0.75))
fig.update_geos(
    projection_type="natural earth",
    showland=True,
    landcolor="lightgray",
    showocean=True,
    oceancolor="lightblue",
    showcountries=True,
    countrycolor="gray",
    coastlinecolor="black"
 )
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

# Gender

In [137]:
#| eval: true
#| echo: false
#| output: true
#| include: true

plots = [px.pie(df, names='Gender'),
        px.pie(df[df['current']==True], names='Gender'),
        px.pie(df[df['alumni']==True], names='Gender')]

fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'domain'}, {'type': 'domain'}, {'type': 'domain'}]], subplot_titles=('Since 2022', 'Current', 'Alumni'))

for i, figure in enumerate(plots):
    for trace in figure.data:
        fig.add_trace(trace, row=1, col=i+1)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))